# Utils - QB - FrequencyAligner

Ce notebook illustre et vérifie le comportement de la classe `FrequencyAligner`
(`tsforecast/frequency/frequency_aligner.py`), qui aligne des colonnes sur une
fréquence cible (agrégation ou interpolation) en préservant les conventions
d'index attendues par `HighFrequencyImputer` : `aggregate_to_target` conserve
l'index d'origine (réindexation, NaN hors bornes), `interpolate_to_target`
densifie l'index (union avec les dates interpolées, restreinte à la plage
d'origine). Les conversions elles-mêmes sont déléguées à
`FrequencyConverter` (`tsforecast/utils/frequency/converter.py`), déjà testée
dans `notebooks/utils/frequency_converter.ipynb` — on ne revérifie pas ici son
comportement interne (positions S/E, `full_periods_only`, etc.), seulement la
façon dont `FrequencyAligner` l'utilise (index, densification, panel).

**Périmètre** : seules les méthodes publiques de `FrequencyAligner` sont testées :
- `extract_column_names(keys)`
- `group_keys_by_entity_and_variable(keys)`
- `get_entity_target_frequency(entity, target_frequency)`
- `get_entity_mask(X, entity)`
- `aggregate_to_target(df, aggregate_keys, target_frequency, is_panel)`
- `restrict_to_original_span(densified_index, original_index)`
- `build_densified_panel_index(df, interpolated)`
- `interpolate_to_target(df, interpolate_keys, target_frequency, is_panel, method, limit, limit_direction, limit_area)`
- `convert_to_target(df, keys, target_frequency, is_panel, ...)`

Les méthodes privées (préfixées `_`, ex. `_target_offset_for_index`,
`_observed_series_for_aggregation`) ne sont pas testées directement : leur
comportement est observé en creux à travers les méthodes publiques qui les
utilisent.

Ce notebook a vocation à servir de base à de futurs tests unitaires
(`tests/frequency/test_frequency_aligner.py`) : les cas limites identifiés
sont signalés par des sections **Point de vigilance**.

## 1 - Import et instanciation

In [ ]:
# Importation des modules
import warnings

import numpy as np
import pandas as pd

# Classe testée
from tsforecast.frequency.frequency_aligner import FrequencyAligner

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore')

# Instanciation de l'aligneur
aligner = FrequencyAligner()

print("FrequencyAligner instancié avec succès !")

## 2 - Jeux de données

### 2.1 - Reprise des jeux de données de `3 - QB - Panel a frequences mixtes heterogene.ipynb`

Les deux fonctions génératrices sont recopiées telles quelles (aucune fonction
partagée n'existe entre notebooks dans ce projet, cf. convention déjà suivie
dans `frequency_converter.ipynb`) pour obtenir :
- `df_timeseries` : indicateurs macroéconomiques mensuels/trimestriels/annuels,
  avec une variable annuelle (`balance_commerciale_annuelle`) dont l'historique
  démarre *avant* la grille mensuelle — l'index global en est irrégulier.
- `df_panel` : panel France/Allemagne/Italie à couverture temporelle
  hétérogène (début/fin propres à chaque pays) et à fréquence de publication
  hétérogène pour `depenses_publiques_pib` (annuelle pour la France et
  l'Italie, trimestrielle pour l'Allemagne) — un cas typique où
  `target_frequency` doit être un dictionnaire par entité.

In [ ]:
# Fonction de création de séries temporelles (recopiée depuis le notebook 3)
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    annual_start_date: str = '2015-01-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.

    Args:
        start_date: Start date for the monthly variables of the dataset.
        end_date: End date for the dataset.
        annual_start_date: Start date for the annual trade balance series, earlier
            than `start_date` so that the resulting temporal index is irregular.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    np.random.seed(seed)

    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    # Production industrielle (mensuelle)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    # Inflation mensuelle (IPC)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    # Taux de chômage (mensuel)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    # PIB trimestriel
    pib_base = 2500
    pib_growth_quarterly = 0.5
    df['pib_trimestriel'] = np.nan
    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    # Balance commerciale annuelle : historique antérieur à la grille mensuelle
    annual_dates = pd.date_range(start=annual_start_date, end=end_date, freq='YS')
    df = df.reindex(df.index.union(annual_dates))
    df.index.name = 'date'

    df['balance_commerciale_annuelle'] = np.nan
    for date in annual_dates:
        year_factor = (date.year - 2018)
        base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
        df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    # Délais de publication
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # Historique limité de la production industrielle
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df


# Création du jeu de données de séries temporelles
df_timeseries = create_timeseries_dataset()

print(f"df_timeseries : {df_timeseries.shape}, période {df_timeseries.index.min().date()} à {df_timeseries.index.max().date()}")
print(f"Colonnes : {list(df_timeseries.columns)}")
df_timeseries.tail(8)

In [ ]:
# Fonction de création d'un jeu de données de panel (recopiée depuis le notebook 3)
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.

    Each entity has its own coverage period (start/end dates) and its own
    publication frequency for the public spending indicator, to simulate a
    heterogeneous panel across entities. The annual trade balance series
    also starts earlier than the other variables for each entity, making
    each entity's temporal index irregular.

    Args:
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    np.random.seed(seed)

    countries = {
        'France': {
            'pib_base': 2800, 'inflation_base': 1.5, 'chomage_base': 8.0, 'depenses_base': 55.0,
            'start_date': '2018-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2018-06-01',
            'depenses_frequency': 'annuelle', 'annual_start_date': '2015-01-01'
        },
        'Allemagne': {
            'pib_base': 3500, 'inflation_base': 1.2, 'chomage_base': 5.5, 'depenses_base': 45.0,
            'start_date': '2018-07-01', 'end_date': '2024-04-01', 'prod_ind_start': '2019-01-01',
            'depenses_frequency': 'trimestrielle', 'annual_start_date': '2016-01-01'
        },
        'Italie': {
            'pib_base': 2200, 'inflation_base': 1.8, 'chomage_base': 10.5, 'depenses_base': 50.0,
            'start_date': '2019-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2019-06-01',
            'depenses_frequency': 'annuelle', 'annual_start_date': '2016-01-01'
        }
    }

    all_data = []
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        infl_trend = np.linspace(
            params['inflation_base'], params['inflation_base'] + np.random.uniform(0.5, 2.0), n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        # Dépenses publiques : fréquence annuelle ou trimestrielle selon le pays
        df_country['depenses_publiques_pib'] = np.nan
        if params['depenses_frequency'] == 'annuelle':
            publication_months = [1]
        else:
            publication_months = [1, 4, 7, 10]

        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        # Balance commerciale annuelle : historique antérieur à la grille mensuelle du pays
        annual_dates = pd.date_range(start=params['annual_start_date'], end=params['end_date'], freq='YS')
        df_country = df_country.reindex(df_country.index.union(annual_dates))
        df_country['country'] = country

        df_country['balance_commerciale_annuelle'] = np.nan
        for date in annual_dates:
            year_factor = (date.year - 2018)
            base = -20 + np.random.uniform(-10, 10) + year_factor * 2
            df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel


# Création du jeu de données de panel
df_panel = create_panel_dataset()

print(f"df_panel : {df_panel.shape}")
print(f"Colonnes : {list(df_panel.columns)}")
print()
print("--- Fréquence de publication de depenses_publiques_pib par entité ---")
for country in df_panel.index.get_level_values('country').unique():
    serie = df_panel.loc[country, 'depenses_publiques_pib'].dropna()
    avg_gap_months = round((serie.index[1:] - serie.index[:-1]).mean().days / 30)
    print(f"  {country} : ~{avg_gap_months} mois entre publications ({len(serie)} observations)")
print()
print("--- Période couverte par entité ---")
for country in df_panel.index.get_level_values('country').unique():
    dates_country = df_panel.loc[country].index
    print(f"  {country} : {dates_country.min().date()} à {dates_country.max().date()} ({len(dates_country)} dates)")

df_panel.loc['Allemagne'].tail(6)

## 7 - `aggregate_to_target(df, aggregate_keys, target_frequency, is_panel)`

**Contrat** : l'index de sortie est toujours identique à celui de `df` — les
valeurs agrégées sont réindexées dessus (NaN hors bornes de période, NaN pour
les périodes incomplètes via `full_periods_only=True`). La fréquence source
est détectée sur les **valeurs observées** de chaque variable (pas sur
l'index), afin qu'une variable trimestrielle portée sur un index mensuel soit
bien traitée comme trimestrielle.

### 7.1 - Séries temporelles : agrégation trimestrielle -> annuelle / mensuelle -> annuelle

In [ ]:
# pib_trimestriel (déjà trimestriel, sur un index mensuel) -> QS : les valeurs
# observées sont détectées comme trimestrielles, pas comme 2/3 de mois manquants
r_pib = aligner._aggregate_to_target(df_timeseries, ['pib_trimestriel'], 'QS', is_panel=False)
print("Index de sortie == index d'entrée ?", r_pib.index.equals(df_timeseries.index))
print("Nb non-NaN avant :", df_timeseries['pib_trimestriel'].notna().sum(), "-> après :", r_pib['pib_trimestriel'].notna().sum())
print(r_pib['pib_trimestriel'].dropna().head(4))

# production_industrielle (mensuelle) -> YS : agrégation par somme
r_prod = aligner._aggregate_to_target(df_timeseries, ['production_industrielle'], 'YS', is_panel=False)
print()
print("production_industrielle agrégée annuellement (sum) :")
print(r_prod['production_industrielle'].dropna().head(3))

### 7.2 - Cas limites : colonne absente, `aggregate_keys` vide, colonne entièrement vide

In [ ]:
# Colonne absente du DataFrame : ignorée silencieusement, DataFrame renvoyé inchangé
r_absent = aligner._aggregate_to_target(df_timeseries, ['colonne_inexistante'], 'QS', is_panel=False)
print("Colonne absente -> DataFrame inchangé :", r_absent.equals(df_timeseries))

# aggregate_keys vide : retourne l'objet df tel quel (même référence)
r_empty_keys = aligner._aggregate_to_target(df_timeseries, [], 'QS', is_panel=False)
print("aggregate_keys=[] -> retourne l'objet original (même identité) :", r_empty_keys is df_timeseries)

# Colonne entièrement NaN : laissée telle quelle (pas d'agrégation possible)
df_avec_nan = df_timeseries.copy()
df_avec_nan['colonne_vide'] = np.nan
r_nan = aligner._aggregate_to_target(df_avec_nan, ['colonne_vide'], 'QS', is_panel=False)
print("Colonne entièrement NaN -> reste entièrement NaN :", r_nan['colonne_vide'].isna().all())

### 7.3 - Point de vigilance : `target_frequency` plus fine que la source

`aggregate_to_target` ne vérifie **pas** que `target_frequency` est bien une
fréquence *plus basse* que la source : c'est `convert_to_target` qui choisit
la bonne méthode (agrégation vs interpolation) en comparant les fréquences
(cf. section 11). Si `aggregate_to_target` est appelé directement avec une
cible plus fine que la source (mésusage), le résultat est silencieusement
incohérent : les sous-périodes vides du calendrier cible sont sommées à `0.0`
au lieu de rester `NaN`, ce qui ne saute pas aux yeux si l'on ne vérifie pas
explicitement les valeurs.

In [ ]:
# balance_commerciale_annuelle (annuelle) agrégée vers QS (plus fin que la source) :
# mésusage de _aggregate_to_target — à ne JAMAIS faire directement, convert_to_target
# est fait pour éviter ce cas (il aurait choisi interpolate_to_target).
r_misuse = aligner._aggregate_to_target(df_timeseries, ['balance_commerciale_annuelle'], 'QS', is_panel=False)
print(r_misuse['balance_commerciale_annuelle'].dropna().head(5))
print()
print("-> Les trimestres sans observation annuelle valent 0.0 (somme d'un bin vide),")
print("   PAS NaN : un piège si _aggregate_to_target est appelé sans passer par convert_to_target.")
assert r_misuse.loc['2018-04-01', 'balance_commerciale_annuelle'] == 0.0

### 7.4 - Panel : `target_frequency` commune à toutes les entités

In [ ]:
# depenses_publiques_pib : fréquence source différente par entité (annuelle FR/IT,
# trimestrielle DE), agrégée à YS pour toutes les entités via une chaîne unique
keys_depenses = [('France', 'depenses_publiques_pib'), ('Allemagne', 'depenses_publiques_pib'), ('Italie', 'depenses_publiques_pib')]
r_panel = aligner._aggregate_to_target(df_panel, keys_depenses, 'YS', is_panel=True)
print("Index de sortie == index d'entrée ?", r_panel.index.equals(df_panel.index))
for country in ['France', 'Allemagne', 'Italie']:
    print(f"  {country} : {r_panel.loc[country, 'depenses_publiques_pib'].dropna().head(3).values}")
print()
print("-> Allemagne (source trimestrielle) : la valeur annuelle est la SOMME des 4 trimestres (~4x la base) ;")
print("   France/Italie (source déjà annuelle) : agrégation annuelle -> annuelle, valeur inchangée.")

### 7.5 - Panel : `target_frequency` en dictionnaire par entité

In [ ]:
# pib_trimestriel agrégé différemment selon l'entité : YS pour France/Italie, QS pour Allemagne
keys_pib = [('France', 'pib_trimestriel'), ('Allemagne', 'pib_trimestriel'), ('Italie', 'pib_trimestriel')]
r_dict = aligner._aggregate_to_target(
    df_panel, keys_pib, {'France': 'YS', 'Allemagne': 'QS', 'Italie': 'YS'}, is_panel=True
)
for country in ['France', 'Allemagne', 'Italie']:
    print(f"  {country} : {r_dict.loc[country, 'pib_trimestriel'].dropna().head(3).values}")

print()
# Entité absente du dictionnaire -> ValueError (propagée depuis get_entity_target_frequency)
try:
    aligner._aggregate_to_target(df_panel, [('France', 'pib_trimestriel')], {'Allemagne': 'QS'}, is_panel=True)
except ValueError as e:
    print("ValueError attendue (entité absente du dict) :", e)

## 10 - `interpolate_to_target(df, interpolate_keys, target_frequency, is_panel, method, limit, limit_direction, limit_area)`

**Contrat** (différent de `aggregate_to_target`) : l'index de sortie est
**densifié** — union de l'index d'origine et des dates générées par
l'interpolation, restreinte à la plage temporelle d'origine. Les colonnes non
interpolées gardent leurs valeurs d'origine et sont NaN sur les nouvelles
dates créées.

### 10.1 - Séries temporelles : densification et colonnes non interpolées

In [ ]:
r_interp = aligner._interpolate_to_target(df_timeseries, ['pib_trimestriel'], 'MS', is_panel=False)
print("Longueur avant :", len(df_timeseries), "-> après :", len(r_interp))
print(r_interp['pib_trimestriel'].dropna().head(6))
print()
print("Index densifié == union(index d'origine, index résultat) ?",
      r_interp.index.equals(df_timeseries.index.union(r_interp.index)))

# Les colonnes NON interpolées gardent leurs valeurs d'origine et sont NaN sur les
# dates nouvellement créées par la densification
print()
print("inflation_ipc : NaN avant =", df_timeseries['inflation_ipc'].isna().sum(),
      "-> NaN après (dates ajoutées incluses) =", r_interp['inflation_ipc'].isna().sum())

### 10.2 - Densification d'une variable annuelle irrégulière (historique antérieur à la grille mensuelle)

In [ ]:
# balance_commerciale_annuelle : 9 observations annuelles, dont certaines antérieures
# à 2018 (début de la grille mensuelle) -> interpolation mensuelle sur toute la plage
r_balance = aligner._interpolate_to_target(df_timeseries, ['balance_commerciale_annuelle'], 'MS', is_panel=False)
print("Nb observations annuelles avant :", df_timeseries['balance_commerciale_annuelle'].notna().sum())
print("Nb observations mensualisées après :", r_balance['balance_commerciale_annuelle'].notna().sum())
print(r_balance['balance_commerciale_annuelle'].dropna().head(14))

### 10.3 - Effet du paramètre `limit`

Par défaut (`limit='default'`), le nombre de NaN consécutifs comblés est
calculé automatiquement à partir du facteur de conversion entre fréquence
source et fréquence cible (ex. 3 pour trimestriel -> mensuel : 2 mois
intermédiaires + 1). Un entier explicite permet de restreindre ce
comblement.

In [ ]:
r_default = aligner._interpolate_to_target(df_timeseries, ['pib_trimestriel'], 'MS', is_panel=False)
r_limit1 = aligner._interpolate_to_target(df_timeseries, ['pib_trimestriel'], 'MS', is_panel=False, limit=1)

print("limit='default' -> nb non-NaN :", r_default['pib_trimestriel'].notna().sum())
print("limit=1         -> nb non-NaN :", r_limit1['pib_trimestriel'].notna().sum())
print()
print(r_limit1['pib_trimestriel'].dropna().head(8))
print("-> Avec limit=1, seul le 1er mois suivant chaque observation trimestrielle est comblé (des trous subsistent).")

### 10.4 - Panel : fréquence source hétérogène par entité, cible commune

In [ ]:
keys_depenses = [('France', 'depenses_publiques_pib'), ('Allemagne', 'depenses_publiques_pib'), ('Italie', 'depenses_publiques_pib')]
r_panel_interp = aligner._interpolate_to_target(df_panel, keys_depenses, 'MS', is_panel=True)
print("Longueur avant :", len(df_panel), "-> après :", len(r_panel_interp))
for country in ['France', 'Allemagne', 'Italie']:
    print(f"  {country} : nb non-NaN = {r_panel_interp.loc[country, 'depenses_publiques_pib'].notna().sum()}, "
          f"exemples = {r_panel_interp.loc[country, 'depenses_publiques_pib'].dropna().head(3).values}")

### 10.5 - Panel : `target_frequency` en dictionnaire par entité

In [ ]:
# Italie a une clé dans keys_depenses mais pas dans le dict de fréquence cible ->
# ValueError propagée par get_entity_target_frequency
try:
    aligner._interpolate_to_target(df_panel, keys_depenses, {'France': 'MS', 'Allemagne': 'MS'}, is_panel=True)
except ValueError as e:
    print("ValueError attendue (Italie absente du dict de cible) :", e)

# Avec les 3 entités couvertes, l'appel aboutit normalement
r_panel_dict = aligner._interpolate_to_target(
    df_panel, keys_depenses, {'France': 'MS', 'Allemagne': 'MS', 'Italie': 'MS'}, is_panel=True
)
print("Toutes entités couvertes -> OK, longueur :", len(r_panel_dict))

### 10.6 - Point de vigilance : `ValueError` reproductible pour certaines combinaisons `target_frequency`

En explorant les combinaisons entité x fréquence cible sur `depenses_publiques_pib`,
`interpolate_to_target` lève une `ValueError: cannot reindex on an axis with
duplicate labels` — **quelle que soit la fréquence source** — dès que
`target_frequency='QS'` est demandé pour une colonne portée sur un DataFrame
dont l'index (avant restriction aux observations) contient déjà des dates
alignées avec des débuts de trimestre (ce qui est le cas ici pour toute
colonne du panel, l'index mensuel MS contenant par construction les 4 dates
QS de chaque année).

Cause probable : contrairement à `aggregate_to_target`, qui restreint la
série aux **valeurs observées** avant de la transmettre au convertisseur
(`_observed_series_for_aggregation`), `interpolate_to_target` transmet la
colonne **complète** (NaN compris) à
`FrequencyConverter.interpolate_to_higher_frequency`. La déduplication interne
de cette dernière échoue alors si le calendrier cible croise l'index source
complet.

Reproductible aussi bien en mode TS qu'en mode panel — ce cas mérite un test
de régression dédié (`tests/frequency/test_frequency_aligner.py`) une fois la
cause confirmée/corrigée.

In [ ]:
# Reproduction minimale, mode séries temporelles : target_frequency='QS' sur une
# colonne (mensuelle ou trimestrielle) du DataFrame complet -> échec systématique
for col in ['balance_commerciale_annuelle', 'pib_trimestriel']:
    try:
        aligner._interpolate_to_target(df_timeseries, [col], 'QS', is_panel=False)
        print(f"{col} -> QS : OK")
    except ValueError as e:
        print(f"{col} -> QS : ValueError - {e}")

print()
# Reproduction en mode panel : la fréquence cible QS échoue pour toutes les entités,
# indépendamment de leur fréquence source (annuelle pour France/Italie, trimestrielle
# pour Allemagne)
for country in ['France', 'Allemagne', 'Italie']:
    try:
        aligner._interpolate_to_target(df_panel, [(country, 'depenses_publiques_pib')], {country: 'QS'}, is_panel=True)
        print(f"{country} -> QS : OK")
    except ValueError as e:
        print(f"{country} -> QS : ValueError - {type(e).__name__}")

print()
# A contrario, target_frequency='MS' fonctionne systématiquement sur les mêmes données
print("Par comparaison, target_frequency='MS' fonctionne pour toutes les entités :")
for country in ['France', 'Allemagne', 'Italie']:
    r = aligner._interpolate_to_target(df_panel, [(country, 'depenses_publiques_pib')], {country: 'MS'}, is_panel=True)
    print(f"  {country} -> MS : OK, {r.loc[country, 'depenses_publiques_pib'].notna().sum()} valeurs non-NaN")

## 11 - `convert_to_target(df, keys, target_frequency, is_panel, ...)`

Méthode générique qui choisit automatiquement, pour chaque clé (et par
entité en mode panel), entre agrégation et interpolation, en comparant la
fréquence source détectée (sur les valeurs observées) et la fréquence cible
via `is_higher_frequency`. C'est la méthode à privilégier en pratique :
`aggregate_to_target`/`interpolate_to_target` appelées directement exposent
aux mésusages illustrés en 7.3 et 10.6.

### 11.1 - Séries temporelles : direction choisie automatiquement, cible commune

In [ ]:
# pib_trimestriel (trimestriel) et balance_commerciale_annuelle (annuelle) sont
# interpolés vers MS (cible plus fine) ; production_industrielle (déjà mensuelle)
# est "agrégée" (source == cible, aggregate_to_lower_frequency avec un seul point par bin)
r_convert = aligner.convert_to_target(
    df_timeseries, ['pib_trimestriel', 'balance_commerciale_annuelle', 'production_industrielle'],
    'MS', is_panel=False
)
print("Longueur avant :", len(df_timeseries), "-> après (densification pour les colonnes interpolées) :", len(r_convert))
print(r_convert[['pib_trimestriel', 'balance_commerciale_annuelle', 'production_industrielle']].dropna().head(3))

### 11.2 - Séries temporelles : cible plus basse que toutes les sources (agrégation pure, pas de densification)

In [ ]:
r_convert_ys = aligner.convert_to_target(df_timeseries, ['pib_trimestriel', 'production_industrielle'], 'YS', is_panel=False)
print("Index inchangé (aggregate_to_target uniquement, pas de densification) ?", r_convert_ys.index.equals(df_timeseries.index))
print(r_convert_ys[['pib_trimestriel', 'production_industrielle']].dropna().head(4))

### 11.3 - Panel : `target_frequency` en dictionnaire, directions mixtes selon l'entité et la colonne

In [ ]:
keys_mixed_panel = [
    ('France', 'depenses_publiques_pib'), ('Allemagne', 'depenses_publiques_pib'), ('Italie', 'depenses_publiques_pib'),
    ('France', 'pib_trimestriel'), ('Allemagne', 'pib_trimestriel'), ('Italie', 'pib_trimestriel'),
]
r_convert_panel = aligner.convert_to_target(
    df_panel, keys_mixed_panel, {'France': 'MS', 'Allemagne': 'MS', 'Italie': 'MS'}, is_panel=True
)
print("Longueur avant :", len(df_panel), "-> après :", len(r_convert_panel))
for country in ['France', 'Allemagne', 'Italie']:
    n_dep = r_convert_panel.loc[country, 'depenses_publiques_pib'].notna().sum()
    n_pib = r_convert_panel.loc[country, 'pib_trimestriel'].notna().sum()
    print(f"  {country} : depenses_publiques_pib non-NaN = {n_dep}, pib_trimestriel non-NaN = {n_pib}")

## 12 - Synthèse

| Méthode | Nature du retour | Index de sortie |
|---|---|---|
| `extract_column_names` | `List[str]` dédupliquée, ordre non garanti | — |
| `group_keys_by_entity_and_variable` | `Dict[tuple, List[str]]`, `()` pour les clés TS | — |
| `get_entity_target_frequency` | `str` (ou `ValueError` si entité absente d'un dict) | — |
| `get_entity_mask` | `np.ndarray[bool]` | — |
| `aggregate_to_target` | `pd.DataFrame` | **identique** à celui de `df` (réindexation, NaN hors bornes/périodes incomplètes) |
| `restrict_to_original_span` | `pd.DatetimeIndex` | borné à `[min, max]` de l'index d'origine |
| `build_densified_panel_index` | `pd.MultiIndex` | union par entité, restreinte à sa plage d'origine |
| `interpolate_to_target` | `pd.DataFrame` | **densifié** (union avec les dates interpolées) |
| `convert_to_target` | `pd.DataFrame` | index d'`aggregate_to_target` et/ou `interpolate_to_target` selon les clés |

**Points de vigilance identifiés (pistes pour `tests/frequency/test_frequency_aligner.py`)** :
1. §7.3 — `aggregate_to_target` appelée avec une cible plus fine que la
   source ne lève aucune erreur et produit des zéros silencieux au lieu de
   NaN sur les sous-périodes vides : à ne jamais utiliser en dehors de
   `convert_to_target`.
2. §10.6 — `interpolate_to_target` lève `ValueError: cannot reindex on an
   axis with duplicate labels` pour `target_frequency='QS'` sur ce jeu de
   données (TS et panel), quelle que soit la fréquence source, alors que
   `'MS'` fonctionne sur les mêmes colonnes. À creuser du côté de
   `FrequencyConverter.interpolate_to_higher_frequency` / du fait que
   `interpolate_to_target` (contrairement à `aggregate_to_target`) ne
   restreint pas la série aux valeurs observées avant conversion.
3. §8 / §9 — une entité absente du dictionnaire `interpolated` dans
   `build_densified_panel_index` (ou plus généralement de tout
   `target_frequency` en dict) conserve ses dates d'origine (§9) mais lève
   `ValueError` si elle est absente d'un dict de fréquence cible passé à
   `aggregate_to_target`/`interpolate_to_target` (§7.5/§10.5) — comportement
   cohérent mais à couvrir explicitement par un test pour chaque méthode.